# Phase 3 – Stress tests and an alternative model

**Paper:** Appiah et al. (2023), *Extremely randomised trees machine learning model for electricity theft detection*, Machine Learning with Applications 12, 100458.

This notebook covers parts (c) and (d) of Phase 3.

For **part (c)** we try a different model, HistGradientBoosting, on the corrected features from Phase 2 and compare it with Extra Trees, both on how well it does and on how long it takes to train.

For **part (d)** we put the authors' model and our corrected model under some pressure to see how they hold up:
1. we take out the strange September 2016 days,
2. we add noise to the meter readings,
3. we flip some of the theft labels,
4. we add a lot of useless features.

Data, cleaning and the 80/20 split (SEED = 42) are the same as in the Phase 2 notebook, so the test customers are exactly the same ones. To run it, put `data.csv` in the same folder as this notebook and run all cells. Everything gets saved to `p3out/`.

In [ ]:
import os
import json
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import sklearn
import imblearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import average_precision_score, roc_auc_score, matthews_corrcoef
from imblearn.combine import SMOTETomek

warnings.filterwarnings("ignore")

SEED = 42
N_JOBS = -1
N_TREES = 200    # Phase 2 used 500, but with this many retrains that takes too long
QUICK = False    # set to True to check the code runs on a small sample first

OUT = "p3out"
os.makedirs(OUT, exist_ok=True)

R = {}   # all scores and timings, saved as json at the end
P = {}   # test probabilities of every model, needed for the bootstrap in section 9

T0 = time.time()


def log(msg):
    # the full run takes about an hour, so we also write progress to a file
    line = f"[{time.time() - T0:7.0f}s] {msg}"
    print(line, flush=True)
    with open(os.path.join(OUT, "progress.txt"), "a") as f:
        f.write(line + "\n")


print("scikit-learn", sklearn.__version__, "| imbalanced-learn", imblearn.__version__, "| cores:", os.cpu_count())

scikit-learn 1.6.1 | imbalanced-learn 0.14.2 | cores: 2


## 1. Loading the data
Same reduction and same split as in Phase 2.

In [ ]:
df = pd.read_csv("data.csv")
y_all = df["FLAG"].to_numpy()

X_raw = df.drop(columns=["CONS_NO", "FLAG"])
X_raw.columns = pd.to_datetime(X_raw.columns, format="%Y/%m/%d")
# make sure every day from Jan 2014 to Oct 2016 is a column, in date order
full_days = pd.date_range("2014-01-01", "2016-10-31")
X_raw = X_raw.reindex(columns=full_days)
A = X_raw.to_numpy(dtype=float)

# drop customers with no real readings (all NaN or 0) and duplicate rows
empty = np.all(np.isnan(A) | (A == 0), axis=1)
dup = pd.DataFrame(A).fillna(-1).duplicated().to_numpy()
keep = ~empty & ~dup
A = A[keep]
y = y_all[keep]

idx = np.arange(len(y))
tr, te = train_test_split(idx, test_size=0.2, stratify=y, random_state=SEED)

if QUICK:
    rng0 = np.random.default_rng(SEED)
    tr = rng0.choice(tr, 4000, replace=False)
    te = rng0.choice(te, 1500, replace=False)

ytr, yte = y[tr], y[te]
print("customers:", len(y), "| train:", len(tr), "| test:", len(te), "| test thieves:", int(yte.sum()))

customers: 40256 | train: 32204 | test: 8052 | test thieves: 716


## 2. The two pipelines
**Authors' model** (the honest version, E2 in Phase 2): gaps filled with Equation 1, values capped at mean + 2σ, Min-Max scaling fitted on the training data, SMOTETomek on the training data only, and then Extra Trees with depth 70 and √p features per split.

**Corrected model** (E3 in Phase 2): our rule-based cleaning, log(1 + x), MAD clipping and 11 summary features, then Extra Trees with the settings that nested CV picked in Phase 2 (max_features = 0.1p, min_samples_leaf = 20, max_depth = 20) and `class_weight="balanced"`.

Both are scored on the test set, which keeps the real ratio of thieves to normal customers. PR-AUC is our main metric and we also report ROC-AUC. For the authors' model we add MCC at the default 0.5 threshold.

In [ ]:
def eq1_fill(X):
    # Equation 1 in the paper: a missing day gets the mean of the day before and the day after,
    # if both are there. Everything else that is missing becomes 0.
    X = X.copy()
    nan = np.isnan(X)
    prev = np.c_[np.full((len(X), 1), np.nan), X[:, :-1]]
    nxt = np.c_[X[:, 1:], np.full((len(X), 1), np.nan)]
    both = nan & ~np.isnan(prev) & ~np.isnan(nxt)
    X[both] = (prev[both] + nxt[both]) / 2
    X[np.isnan(X)] = 0.0
    return X


def cap_2sigma(X):
    # anything above mean + 2 std (per customer) is cut down to that value
    mu = X.mean(axis=1, keepdims=True)
    sd = X.std(axis=1, keepdims=True)
    return np.minimum(X, mu + 2 * sd)


def nan_runs(nan_row):
    # start and end index of each block of consecutive missing days
    d = np.diff(np.r_[0, nan_row.astype(int), 0])
    return np.where(d == 1)[0], np.where(d == -1)[0]


def longest_run(nan_row):
    starts, ends = nan_runs(nan_row)
    return int((ends - starts).max()) if len(starts) else 0


def interp_short_gaps(X, max_gap=3):
    # linear interpolation, but only for gaps of max_gap days or less.
    # gaps touching the start or end of the series are left as they are
    X = X.copy()
    for r in X:
        nan = np.isnan(r)
        if not nan.any():
            continue
        starts, ends = nan_runs(nan)
        for a, b in zip(starts, ends):
            if b - a <= max_gap and a > 0 and b < len(r):
                r[a:b] = np.linspace(r[a - 1], r[b], b - a + 2)[1:-1]
    return X


def corrected_features(Araw, days):
    # the E3 features from Phase 2: cleaned log readings + 11 summary features
    C = Araw.copy()
    C[C > 10000] = np.nan      # not physically possible, treat as missing
    nan0 = np.isnan(C)         # missing days before interpolation, used for the summary features
    C = interp_short_gaps(C, 3)
    L = np.log1p(C)

    with np.errstate(all="ignore"):
        # robust centre and spread of each customer on the log scale
        med = np.nanmedian(L, axis=1, keepdims=True)
        mad = np.nanmedian(np.abs(L - med), axis=1, keepdims=True) * 1.4826
        hi = med + 3 * mad
        lo = med - 3 * mad
        n_hi = np.nansum(L > hi, axis=1)
        n_lo = np.nansum(L < lo, axis=1)
        L = np.where(L > hi, hi, L)   # only the high side is clipped, low values may be the theft itself

        obs = ~np.isnan(C)
        in_2016 = np.asarray(days >= pd.Timestamp("2016-01-01"))
        summary = np.c_[
            nan0.mean(axis=1),                                            # share of missing days
            [longest_run(r) for r in nan0],                               # longest gap
            np.argmax(obs, axis=1),                                       # first day with a reading
            nan0[:, in_2016].mean(axis=1),                                # share missing in 2016
            np.nansum(C == 0, axis=1) / np.maximum(obs.sum(axis=1), 1),   # share of zero readings
            med[:, 0],
            mad[:, 0],
            np.nanstd(C, axis=1) / np.nanmean(C, axis=1),                 # coefficient of variation
            np.nanmedian(L[:, -90:], axis=1) - med[:, 0],                 # last 90 days vs usual level
            n_hi,
            n_lo,
        ]
    summary[~np.isfinite(summary)] = np.nan
    return np.hstack([L, summary]).astype(np.float32)


def authors_train(Atr, ytr_):
    # the authors' steps, but fitted on the training data only (E2)
    Xa = cap_2sigma(eq1_fill(Atr))
    sc = MinMaxScaler().fit(Xa)
    Xs = sc.transform(Xa).astype(np.float32)
    Xr, yr = SMOTETomek(random_state=SEED).fit_resample(Xs, ytr_)
    model = ExtraTreesClassifier(n_estimators=N_TREES, max_depth=70, random_state=SEED, n_jobs=N_JOBS)
    model.fit(Xr, yr)
    return model, sc


def authors_X(Ate, sc):
    return sc.transform(cap_2sigma(eq1_fill(Ate))).astype(np.float32)


def corrected_model():
    return ExtraTreesClassifier(n_estimators=N_TREES, max_features=0.1, min_samples_leaf=20, max_depth=20,
                                class_weight="balanced", random_state=SEED, n_jobs=N_JOBS)


def score(y_true, p, thr=None):
    out = {"PR_AUC": float(average_precision_score(y_true, p)),
           "ROC_AUC": float(roc_auc_score(y_true, p))}
    if thr is not None:
        out["MCC"] = float(matthews_corrcoef(y_true, (p >= thr).astype(int)))
    return out


def show(name, s):
    print(f"{name:55s} " + " | ".join(f"{k} {v:.4f}" for k, v in s.items()), flush=True)

## 3. Baselines (no stress)
First both models are trained on the normal data. All the stress tests below are compared against these numbers.

In [ ]:
log("baseline: authors' model")
t = time.time()
auth, sc = authors_train(A[tr], ytr)
R["time_authors_train_s"] = time.time() - t
p_auth = auth.predict_proba(authors_X(A[te], sc))[:, 1]

log("baseline: corrected features")
X3 = corrected_features(A, full_days)
X3tr, X3te = X3[tr], X3[te]
t = time.time()
cor = corrected_model().fit(X3tr, ytr)
R["time_corrected_ET_train_s"] = time.time() - t
p_cor = cor.predict_proba(X3te)[:, 1]

R["baseline"] = {"authors": score(yte, p_auth, 0.5), "corrected": score(yte, p_cor)}
P["baseline|authors"] = p_auth
P["baseline|corrected"] = p_cor
show("Baseline authors' model", R["baseline"]["authors"])
show("Baseline corrected Extra Trees", R["baseline"]["corrected"])

[     17s] baseline: authors' model
[    407s] baseline: corrected features
Baseline authors' model                                 PR_AUC 0.3358 | ROC_AUC 0.7775 | MCC 0.2750
Baseline corrected Extra Trees                          PR_AUC 0.3782 | ROC_AUC 0.8000


## 4. Part (c): alternative model – HistGradientBoosting
Gradient boosting adds trees one at a time, and each new tree tries to fix the mistakes of the ones before it. scikit-learn's HistGradientBoosting first puts every feature into at most 255 bins, which makes it fast on a dataset this size, and it deals with missing values by itself so we don't need to fill them in.

It gets the same corrected features and class weights as the Extra Trees model. The number of trees is decided by early stopping on 10% of the training data.

In [ ]:
log("alternative: HistGradientBoosting")
hgb = HistGradientBoostingClassifier(learning_rate=0.05, max_iter=500, max_leaf_nodes=31, min_samples_leaf=50,
                                     l2_regularization=1.0, class_weight="balanced", early_stopping=True,
                                     validation_fraction=0.1, n_iter_no_change=30, random_state=SEED)
t = time.time()
hgb.fit(X3tr, ytr)
R["time_hgb_train_s"] = time.time() - t
p_hgb = hgb.predict_proba(X3te)[:, 1]

R["hgb"] = score(yte, p_hgb)
R["hgb"]["n_iter"] = int(hgb.n_iter_)
P["baseline|hgb"] = p_hgb
show("HistGradientBoosting (corrected features)", R["hgb"])

print(f"training time: corrected Extra Trees {R['time_corrected_ET_train_s']:.0f}s | "
      f"HistGradientBoosting {R['time_hgb_train_s']:.0f}s | authors' pipeline {R['time_authors_train_s']:.0f}s")

# how many of the top 5% most suspicious customers are actually thieves,
# i.e. roughly the hit rate an inspection team would get
k = max(1, int(round(0.05 * len(yte))))
R["top5"] = {name: float(yte[np.argsort(-p)[:k]].mean())
             for name, p in [("authors", p_auth), ("corrected", p_cor), ("hgb", p_hgb)]}
print("precision in top 5%:", {name: round(v, 3) for name, v in R["top5"].items()})

[    681s] alternative: HistGradientBoosting
HistGradientBoosting (corrected features)               PR_AUC 0.4549 | ROC_AUC 0.8248 | n_iter 180.0000
training time: corrected Extra Trees 260s | HistGradientBoosting 86s | authors' pipeline 389s
precision in top 5%: {'authors': 0.424, 'corrected': 0.509, 'hgb': 0.566}


## 5. Stress test 1: removing the September 2016 days
In Phase 2 we found that the corrected model relies a lot on 15–21 September 2016. In that week about 20% of the thieves have no readings, compared with only around 6% of the normal customers. So here we cut 10–25 September 2016 out of both the training and the test data and retrain both models.

**Hypothesis:** if the models mainly picked up this quirk in how the data was collected, and not real theft behaviour, their scores should drop clearly.

In [ ]:
log("stress 1: remove 10-25 Sep 2016")
sep_days = (full_days >= pd.Timestamp("2016-09-10")) & (full_days <= pd.Timestamp("2016-09-25"))
A1 = A[:, ~sep_days]
days1 = full_days[~sep_days]

auth1, sc1 = authors_train(A1[tr], ytr)
pa1 = auth1.predict_proba(authors_X(A1[te], sc1))[:, 1]

X31 = corrected_features(A1, days1)
cor1 = corrected_model().fit(X31[tr], ytr)
pc1 = cor1.predict_proba(X31[te])[:, 1]

R["stress_sep"] = {"days_removed": int(sep_days.sum()),
                   "authors": score(yte, pa1, 0.5),
                   "corrected": score(yte, pc1)}
P["sep|authors"] = pa1
P["sep|corrected"] = pc1
show("Sep removed: authors", R["stress_sep"]["authors"])
show("Sep removed: corrected", R["stress_sep"]["corrected"])

[    768s] stress 1: remove 10-25 Sep 2016
Sep removed: authors                                    PR_AUC 0.3300 | ROC_AUC 0.7760 | MCC 0.2740
Sep removed: corrected                                  PR_AUC 0.3580 | ROC_AUC 0.7947


## 6. Stress test 2: noisy meter readings
Meters are never perfectly accurate. Here we take the models already trained in section 3 and give them test readings with random noise added: each reading x becomes x · e^(σ·z), where z ~ N(0, 1). σ = 0.1 is roughly ±10% error and σ = 0.5 roughly ±50%.

Nothing is retrained. The point is to see what happens when the meters out in the field are noisier than the data the model was trained on.

**Hypothesis:** a bit of noise shouldn't change much, because trees only care about the order of the values and not the exact numbers, and the pattern of missing days stays the same. With a lot of noise the scores should go down.

In [ ]:
R["stress_noise"] = {}
for sigma in [0.1, 0.3, 0.5]:
    log(f"stress 2: reading noise sigma={sigma}")
    # same seed each time, so z is identical and only the size of the noise changes
    rng = np.random.default_rng(SEED)
    An = A[te] * np.exp(sigma * rng.standard_normal(A[te].shape))   # noise on the test readings only

    pa = auth.predict_proba(authors_X(An, sc))[:, 1]
    pc = cor.predict_proba(corrected_features(An, full_days))[:, 1]

    res = {"authors": score(yte, pa, 0.5), "corrected": score(yte, pc)}
    R["stress_noise"][str(sigma)] = res
    P[f"noise{sigma}|authors"] = pa
    P[f"noise{sigma}|corrected"] = pc
    show(f"noise {sigma}: authors", res["authors"])
    show(f"noise {sigma}: corrected", res["corrected"])

[   1406s] stress 2: reading noise sigma=0.1
noise 0.1: authors                                      PR_AUC 0.3346 | ROC_AUC 0.7754 | MCC 0.2770
noise 0.1: corrected                                    PR_AUC 0.3756 | ROC_AUC 0.8001
[   1411s] stress 2: reading noise sigma=0.3
noise 0.3: authors                                      PR_AUC 0.3224 | ROC_AUC 0.7587 | MCC 0.2759
noise 0.3: corrected                                    PR_AUC 0.3684 | ROC_AUC 0.7960
[   1416s] stress 2: reading noise sigma=0.5
noise 0.5: authors                                      PR_AUC 0.3051 | ROC_AUC 0.7391 | MCC 0.2714
noise 0.5: corrected                                    PR_AUC 0.3586 | ROC_AUC 0.7847


## 7. Stress test 3: wrong labels
The theft labels come from inspections, and some thieves never get caught, so in practice a few of the "normal" customers are really thieves. To imitate this we flip 10% and then 20% of the theft labels in the training data to normal and retrain both models. The test labels stay as they are.

**Hypothesis:** both models get somewhat worse. The authors' model could be hit harder, because SMOTE builds a lot of synthetic thieves out of the (now fewer) real ones.

In [ ]:
R["stress_labels"] = {}
for frac in [0.1, 0.2]:
    log(f"stress 3: flip {frac:.0%} of training theft labels")
    rng = np.random.default_rng(SEED)
    yn = ytr.copy()
    thieves = np.where(yn == 1)[0]
    flipped = rng.choice(thieves, int(frac * len(thieves)), replace=False)
    yn[flipped] = 0

    a_m, a_sc = authors_train(A[tr], yn)
    pa = a_m.predict_proba(authors_X(A[te], a_sc))[:, 1]
    pc = corrected_model().fit(X3tr, yn).predict_proba(X3te)[:, 1]

    res = {"authors": score(yte, pa, 0.5), "corrected": score(yte, pc)}
    R["stress_labels"][str(frac)] = res
    P[f"labels{frac}|authors"] = pa
    P[f"labels{frac}|corrected"] = pc
    show(f"label noise {frac}: authors", res["authors"])
    show(f"label noise {frac}: corrected", res["corrected"])

[   1420s] stress 3: flip 10% of training theft labels
label noise 0.1: authors                                PR_AUC 0.3231 | ROC_AUC 0.7717 | MCC 0.2713
label noise 0.1: corrected                              PR_AUC 0.3731 | ROC_AUC 0.8002
[   2057s] stress 3: flip 20% of training theft labels
label noise 0.2: authors                                PR_AUC 0.3227 | ROC_AUC 0.7704 | MCC 0.2789
label noise 0.2: corrected                              PR_AUC 0.3730 | ROC_AUC 0.7964


## 8. Stress test 4: more dimensions
Here we add k useless features to every customer. Each one is a copy of a real day column with its values shuffled between customers, so it looks like normal consumption data but has nothing to do with the label. k = 1,000 roughly doubles the number of features, and with k = 4,000 about 80% of the features are useless.

For the authors' model the extra columns are added after SMOTETomek (and shuffled the same way), so the resampling doesn't have to run on the much bigger data.

**Hypothesis:** the authors' Extra Trees only tries √p random features at each split, so as p grows most of what it gets to choose from is junk and its scores should drop. The corrected model should lose a bit as well. HistGradientBoosting checks every feature at every split, so it should hold up best.

In [ ]:
def add_noise_features(X, cols, rng):
    # copy the chosen day columns and shuffle each one across customers, so there's no signal left.
    # train and test get the same cols, so fake feature j is the same day in both
    extra = np.empty((len(X), len(cols)), dtype=np.float32)
    for j, c in enumerate(cols):
        extra[:, j] = X[rng.permutation(len(X)), c]
    return np.hstack([X, extra])


# authors' data is resampled once here, the fake columns get added after that
Xa_tr = cap_2sigma(eq1_fill(A[tr]))
scd = MinMaxScaler().fit(Xa_tr)
Xs_tr = scd.transform(Xa_tr).astype(np.float32)
Xs_te = authors_X(A[te], scd)
Xr_d, yr_d = SMOTETomek(random_state=SEED).fit_resample(Xs_tr, ytr)

R["stress_dims"] = {}
for k_add in [1000, 4000]:
    log(f"stress 4: add {k_add} useless features")
    rng = np.random.default_rng(SEED)
    cols = rng.integers(0, 1035, k_add)   # columns 0-1034 are the daily readings in both feature sets

    Xr_k = add_noise_features(Xr_d, cols, rng)
    Xte_k = add_noise_features(Xs_te, cols, rng)
    m = ExtraTreesClassifier(n_estimators=N_TREES, max_depth=70, random_state=SEED, n_jobs=N_JOBS).fit(Xr_k, yr_d)
    pa = m.predict_proba(Xte_k)[:, 1]
    del Xr_k   # this one is huge after SMOTE, free it before the next step

    X3tr_k = add_noise_features(X3tr, cols, rng)
    X3te_k = add_noise_features(X3te, cols, rng)
    pc = corrected_model().fit(X3tr_k, ytr).predict_proba(X3te_k)[:, 1]
    ph = HistGradientBoostingClassifier(**hgb.get_params()).fit(X3tr_k, ytr).predict_proba(X3te_k)[:, 1]

    res = {"authors": score(yte, pa, 0.5), "corrected": score(yte, pc), "hgb": score(yte, ph)}
    R["stress_dims"][str(k_add)] = res
    P[f"dims{k_add}|authors"] = pa
    P[f"dims{k_add}|corrected"] = pc
    P[f"dims{k_add}|hgb"] = ph
    for name in ["authors", "corrected", "hgb"]:
        show(f"+{k_add} features: {name}", res[name])
    del X3tr_k, X3te_k

[   2960s] stress 4: add 1000 useless features
+1000 features: authors                                 PR_AUC 0.3105 | ROC_AUC 0.7700 | MCC 0.2769
+1000 features: corrected                               PR_AUC 0.3644 | ROC_AUC 0.7942
+1000 features: hgb                                     PR_AUC 0.3927 | ROC_AUC 0.7979
[   3830s] stress 4: add 4000 useless features
+4000 features: authors                                 PR_AUC 0.2975 | ROC_AUC 0.7625 | MCC 0.2580
+4000 features: corrected                               PR_AUC 0.3461 | ROC_AUC 0.7786
+4000 features: hgb                                     PR_AUC 0.3860 | ROC_AUC 0.7994


## 9. Real change or just luck? (paired bootstrap)
Every stress result above comes from one run on one test set, so a small difference could just be chance. To check this we resample the 8,052 test customers with replacement 1,000 times. The resampling is stratified, so every sample still has 716 thieves.

In each sample we compute the PR-AUC of the stressed model and of its no-stress baseline on the same customers, and store the difference. The 2.5th and 97.5th percentiles of those differences give a 95% confidence interval. If the interval doesn't include 0, the change probably isn't down to which customers happened to end up in the test set.

One thing to keep in mind: this only covers the test-set sampling. Randomness in training (for example a different seed for the trees) isn't included.

In [ ]:
log("bootstrap confidence intervals")
# keep the probabilities so the bootstrap can be redone later without retraining everything
np.savez_compressed(f"{OUT}/test_probs.npz", y_test=yte, **{k.replace("|", "__"): v for k, v in P.items()})

B_REPS = 200 if QUICK else 1000
rng_b = np.random.default_rng(SEED)
pos_i = np.where(yte == 1)[0]
neg_i = np.where(yte == 0)[0]
# the same bootstrap samples are used for every comparison (that's what makes it paired)
boots = [np.r_[rng_b.choice(pos_i, len(pos_i)), rng_b.choice(neg_i, len(neg_i))] for _ in range(B_REPS)]


def boot_diff(p_new, p_base):
    d = [average_precision_score(yte[b], p_new[b]) - average_precision_score(yte[b], p_base[b]) for b in boots]
    return float(np.mean(d)), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))


CI = {}
for key in P:
    test_name, model = key.split("|")
    if test_name != "baseline":
        CI[key] = boot_diff(P[key], P[f"baseline|{model}"])
CI["hgb_vs_corrected"] = boot_diff(P["baseline|hgb"], P["baseline|corrected"])
CI["corrected_vs_authors"] = boot_diff(P["baseline|corrected"], P["baseline|authors"])

R["bootstrap_PR_AUC_change"] = {k: {"mean": v[0], "ci_low": v[1], "ci_high": v[2]} for k, v in CI.items()}
for key, (mean, lo, hi) in CI.items():
    verdict = "significant" if lo > 0 or hi < 0 else "not significant"
    print(f"{key:28s} change {mean:+.4f}   95% CI [{lo:+.4f}, {hi:+.4f}]   {verdict}")

## 10. Summary table and figure
All the results, with their bootstrap intervals, go into `stress_results.csv`, and the PR-AUC of each test is plotted in `fig_stress.png`.

In [ ]:
rows = []


def add_row(test, setting, model, s, ci_key=None):
    row = {"test": test, "setting": setting, "model": model, **s}
    if ci_key is not None:
        mean, lo, hi = CI[f"{ci_key}|{model}"]
        row.update(PR_AUC_change=mean, ci_low=lo, ci_high=hi)
    rows.append(row)


for n in ["authors", "corrected"]:
    add_row("baseline", "none", n, R["baseline"][n])
add_row("baseline", "none", "hgb", R["hgb"])

for n in ["authors", "corrected"]:
    add_row("sep_removed", "10-25 Sep 2016", n, R["stress_sep"][n], "sep")

for sigma, res in R["stress_noise"].items():
    for n in ["authors", "corrected"]:
        add_row("reading_noise", f"sigma={sigma}", n, res[n], f"noise{sigma}")

for frac, res in R["stress_labels"].items():
    for n in ["authors", "corrected"]:
        add_row("label_noise", f"{float(frac):.0%} flipped", n, res[n], f"labels{frac}")

for k_add, res in R["stress_dims"].items():
    for n in ["authors", "corrected", "hgb"]:
        add_row("extra_features", f"+{k_add}", n, res[n], f"dims{k_add}")

tab = pd.DataFrame(rows)
print(tab.round(4).to_string(index=False))
tab.to_csv(f"{OUT}/stress_results.csv", index=False)
with open(f"{OUT}/results_phase3.json", "w") as f:
    json.dump(R, f, indent=1)


# figure: PR-AUC of every model under each stress test
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 200})
colors = {"authors": "#D9534F", "corrected": "#2A78D6", "hgb": "#54A24B"}
names = {"authors": "Authors' model", "corrected": "Corrected Extra Trees", "hgb": "HistGradientBoosting"}


def base(n):
    return (R["hgb"] if n == "hgb" else R["baseline"][n])["PR_AUC"]


def plot_panel(ax, xs, get_score, models, title, xlabel):
    for n in models:
        ax.plot(range(len(xs)), [get_score(x, n) for x in xs], "o-", color=colors[n], label=names[n])
    ax.set_xticks(range(len(xs)))
    ax.set_xticklabels([str(x) for x in xs])
    ax.set_title(title, loc="left", fontsize=10)
    ax.set_xlabel(xlabel)


fig, axes = plt.subplots(1, 4, figsize=(12, 3.4), sharey=True)
plot_panel(axes[0], ["base", "removed"],
           lambda x, n: base(n) if x == "base" else R["stress_sep"][n]["PR_AUC"],
           ["authors", "corrected"], "a) Sep 2016 days removed", "")
plot_panel(axes[1], [0, 0.1, 0.3, 0.5],
           lambda x, n: base(n) if x == 0 else R["stress_noise"][str(x)][n]["PR_AUC"],
           ["authors", "corrected"], "b) Noise on readings", "σ")
plot_panel(axes[2], [0, 0.1, 0.2],
           lambda x, n: base(n) if x == 0 else R["stress_labels"][str(x)][n]["PR_AUC"],
           ["authors", "corrected"], "c) Theft labels flipped", "share flipped")
plot_panel(axes[3], [0, 1000, 4000],
           lambda x, n: base(n) if x == 0 else R["stress_dims"][str(x)][n]["PR_AUC"],
           ["authors", "corrected", "hgb"], "d) Useless features added", "features added")

axes[0].set_ylabel("Test PR-AUC")
axes[3].legend(frameon=False, fontsize=8)
for ax in axes:
    ax.axhline(yte.mean(), color="#999", ls="--", lw=0.8)   # what random guessing would get
plt.tight_layout()
plt.savefig(f"{OUT}/fig_stress.png")
plt.close()
log("done")